# 11.4 · 文本分类 / Text Classification

> **课程定位 / Where this fits**
> 第 4 课，**Part 11 · 经典 NLP**。最常见、最实用的 NLP 任务。
> Lesson 4, **Part 11 · Classic NLP**. The most common, practical NLP task.
>
> 有了文本表示(TF-IDF / 词向量)，就能做**文本分类**：判断一段文本属于哪一类——垃圾邮件识别、新闻主题分类、情感判断、意图识别、内容审核……都是它。本课用 **TF-IDF + 逻辑回归 / 朴素贝叶斯** 搭建**强基线**(至今很多生产系统仍在用)，并讲清**评估**(混淆矩阵、各类 P/R/F1)、**可解释性**(每类最具区分力的词)和**误差分析**——这套流程是 NLP 工程的日常。
> With text representations (TF-IDF / embeddings), we can do **text classification**: assign a text to a category — spam detection, news topics, sentiment, intent, moderation. We build **strong baselines** with **TF-IDF + logistic regression / naive Bayes** (still used in production), plus **evaluation** (confusion matrix, per-class P/R/F1), **interpretability** (most discriminative words per class), and **error analysis** — the bread and butter of NLP engineering.
>
> 💼 **实战/面试视角**："TF-IDF+线性模型为何是强基线 / 朴素贝叶斯假设 / 类别不平衡 / 如何做误差分析" 高频实战。
> 💼 **Practical/interview angle:** "why TF-IDF+linear is a strong baseline / naive Bayes assumption / class imbalance / error analysis" — frequent and practical.

> 📐 **符号约定 / Notation**
> - 管道(pipeline) —— 把向量化+分类器串成一个可复用流程 / vectorizer + classifier chained
> - P/R/F1 —— 精确率/召回率/F1(见 Part 7) / precision/recall/F1

> 💡 **面试相关 / Interview-relevant**
> - "文本分类的标准流程"（出镜率 ★★★★）
> - "朴素贝叶斯的'朴素'假设是什么"（★★★★★）
> - "为什么 TF-IDF + 线性模型是强基线"（★★★★）
> - "如何解释模型/做误差分析"（★★★★）

---

## 学习目标 / Learning Objectives
1. 掌握文本分类标准流程(表示→分类器→评估)。
   Master the text-classification pipeline (represent → classify → evaluate).
2. 用 **TF-IDF + 朴素贝叶斯 / 逻辑回归** 搭基线并对比。
   Build & compare TF-IDF + naive Bayes / logistic regression baselines.
3. 用**混淆矩阵 + 分类报告**评估多分类。
   Evaluate multi-class with confusion matrix + classification report.
4. 解释模型(**每类最具区分力的词**)并做**误差分析**。
   Interpret the model (top words per class) and do error analysis.

## 目录 / TOC
1. [任务与数据 ⭐](#1)
2. [基线：TF-IDF + 分类器 ⭐](#2)
3. [评估：混淆矩阵与分类报告 ⭐](#3)
4. [可解释性与误差分析 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 任务与数据 ⭐ / Task & Data

**文本分类**：给定一段文本，预测它的**类别标签**。我们用 **20 Newsgroups** 的 4 个类别(无神论、计算机图形、太空、棒球)做**多分类**。
**Text classification:** given a text, predict its **category**. We use 4 categories of **20 Newsgroups** (atheism, computer graphics, space, baseball) for **multi-class** classification.

标准流程：**原始文本 → (预处理+向量化) → 分类器 → 预测类别**。前两步我们已在 11.1/11.2 学过，这里把它们和分类器串成一个**管道(pipeline)**。
Standard pipeline: **raw text → (preprocess + vectorize) → classifier → predicted class**. We learned the first steps in 11.1/11.2; here we chain them with a classifier into a **pipeline**.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import fetch_20newsgroups
sns.set_theme(style="whitegrid")

cats = ["alt.atheism", "comp.graphics", "sci.space", "rec.sport.baseball"]
train = fetch_20newsgroups(subset="train", categories=cats, remove=("headers","footers","quotes"))
test  = fetch_20newsgroups(subset="test",  categories=cats, remove=("headers","footers","quotes"))
names = [c.split(".")[-1] for c in train.target_names]
print(f"训练 {len(train.data)} 篇, 测试 {len(test.data)} 篇, {len(cats)} 类: {names}")

fig, ax = plt.subplots(figsize=(7,3.2))
counts = np.bincount(train.target)
ax.bar(names, counts, color="#39c")
for i,v in enumerate(counts): ax.text(i, v+3, str(v), ha="center")
ax.set_ylabel("训练文档数"); ax.set_title("各类别样本数(大致均衡)")
plt.tight_layout(); plt.show()
print("\n一篇示例(太空类节选):")
print(train.data[np.where(train.target==2)[0][0]][:200])


<a id="2"></a>
## 2. 基线：TF-IDF + 分类器 ⭐ / Baseline: TF-IDF + Classifier

把 **TF-IDF 向量化器** 和 **分类器** 用 `Pipeline` 串起来——一个对象就能 `fit`/`predict`，避免训练/测试时手动重复向量化(还能防数据泄漏，呼应 Part 3)。
Chain a **TF-IDF vectorizer** and a **classifier** with `Pipeline` — one object to `fit`/`predict`, avoiding manual re-vectorization and preventing leakage (echoing Part 3).

对比两个经典基线：
Compare two classic baselines:
- **朴素贝叶斯(Multinomial NB)**：基于贝叶斯定理，**"朴素"假设 = 给定类别下各词相互独立**(忽略词之间的关系)。这个假设明显不真，但对文本**出奇地好用**、极快，是经典基线。
  **Multinomial Naive Bayes:** Bayes' theorem with the **"naive" assumption = words are conditionally independent given the class** (ignores word relations). Clearly false, yet **surprisingly effective** for text and very fast.
- **逻辑回归(Logistic Regression)**：直接学每个词对每个类别的**权重**，通常精度更高、可解释。
  **Logistic Regression:** learns a **weight** for each word→class; usually higher accuracy and interpretable.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def make_pipe(clf):
    return Pipeline([("tfidf", TfidfVectorizer(stop_words="english", min_df=3, ngram_range=(1,2))),  # 向量化 / vectorize
                     ("clf", clf)])                                                                    # 分类器 / classifier

models = {"朴素贝叶斯 NB": MultinomialNB(),
          "逻辑回归 LogReg": LogisticRegression(max_iter=1000, C=10)}
results = {}
for name, clf in models.items():
    pipe = make_pipe(clf); pipe.fit(train.data, train.target)          # 一步训练(自动向量化+拟合) / fit end-to-end
    acc = accuracy_score(test.target, pipe.predict(test.data))         # 测试集准确率 / test accuracy
    results[name] = (pipe, acc)
    print(f"{name}: test 准确率 = {acc:.3f}")
best_name = max(results, key=lambda k: results[k][1]); best_pipe = results[best_name][0]
print(f"\n最佳: {best_name}; TF-IDF+线性模型就是很强的基线(简单/快/可解释)")


<a id="3"></a>
## 3. 评估：混淆矩阵与分类报告 ⭐ / Evaluation: Confusion Matrix & Report

只看准确率不够(呼应 Part 7)。**混淆矩阵**显示"哪类被错分成哪类"，**分类报告**给出每类的精确率/召回率/F1。
Accuracy alone isn't enough (echoing Part 7). The **confusion matrix** shows "which class is mistaken for which," and the **classification report** gives per-class precision/recall/F1.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
pred = best_pipe.predict(test.data)
cm = confusion_matrix(test.target, pred)
fig, ax = plt.subplots(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
ax.set_xlabel("预测类别"); ax.set_ylabel("真实类别"); ax.set_title(f"混淆矩阵 ({best_name})")
plt.tight_layout(); plt.show()
print(classification_report(test.target, pred, target_names=names, digits=3))
print("对角线=正确; 看非对角线找混淆最多的类对(如 atheism 易与 graphics 混? 看具体矩阵)")


<a id="4"></a>
## 4. 可解释性与误差分析 + 小结 ⭐ / Interpretability & Error Analysis

线性模型的一大优点是**可解释**：逻辑回归对每个(词, 类别)有一个权重，权重最高的词就是该类别**最具区分力的特征**。这能帮你**验证模型是否学到合理信号**(而非数据泄漏的伪特征)。
A big advantage of linear models is **interpretability**: logistic regression has a weight per (word, class); the highest-weight words are a class's **most discriminative features**. This helps **verify the model learned sensible signals** (not leaked spurious features).


In [ ]:
# 每个类别权重最高的词 = 最具区分力的特征 / top weighted words per class
lr_pipe = results["逻辑回归 LogReg"][0]
vec = lr_pipe.named_steps["tfidf"]; clf = lr_pipe.named_steps["clf"]
feat = np.array(vec.get_feature_names_out())
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ci, ax in enumerate(axes):
    top = clf.coef_[ci].argsort()[-8:]                    # 该类权重最高的8个词 / top-8 words for this class
    ax.barh(feat[top], clf.coef_[ci][top], color="#2a9d8f")
    ax.set_title(names[ci], fontsize=10)
fig.suptitle("每个类别最具区分力的词(逻辑回归权重) → 验证模型学到合理信号"); plt.tight_layout(); plt.show()
print("space类→orbit/space/nasa, baseball类→game/team/year... → 模型学到了符合直觉的关键词")


In [ ]:
# 误差分析: 看几个被分错的样本 / error analysis: inspect misclassified examples
wrong = np.where(pred != test.target)[0]
print(f"误分类样本数: {len(wrong)} / {len(test.target)}\n")
for i in wrong[:3]:
    print(f"真实={names[test.target[i]]:10} 预测={names[pred[i]]:10}")
    print(f"  文本节选: {test.data[i][:120].strip()!r}")
    print()
print("误差分析价值: 发现是标注噪声? 类别本身相近? 还是模型缺特征? → 指导下一步改进")
print("常见改进: 调 ngram/min_df, 换模型, 加特征, 处理类别不平衡(class_weight), 清洗数据")


```
流程: 原始文本→(预处理+TF-IDF向量化)→分类器→预测; 用 Pipeline 串起来(防泄漏/可复用)
基线: TF-IDF + 朴素贝叶斯(快/'朴素'=词条件独立假设) 或 逻辑回归(更准/可解释)
为什么强基线: 文本TF-IDF高维线性可分, 线性模型简单快可解释, 生产至今常用
评估: 不只准确率; 混淆矩阵(谁错成谁) + 各类 P/R/F1(Part 7)
可解释: 逻辑回归每(词,类)权重 → 最具区分力的词; 验证学到合理信号
误差分析: 看错分样本→定位原因(标注噪声/类别相近/缺特征)→指导改进
```

### 💡 面试速查 / Interview cheat-sheet
1. **标准流程**: 文本→TF-IDF→线性分类器, Pipeline 封装。
   Pipeline: text→TF-IDF→linear classifier, wrapped in a Pipeline.
2. **朴素贝叶斯**: 给定类别下词条件独立(假设不真但好用/快)。
   Naive Bayes: words conditionally independent given class (false but effective/fast).
3. **强基线**: TF-IDF+线性模型简单/快/可解释, 难被轻易超越。
   Strong baseline: TF-IDF+linear is simple/fast/interpretable, hard to beat easily.
4. **评估**: 混淆矩阵 + 各类 P/R/F1, 别只看准确率。
   Evaluation: confusion matrix + per-class P/R/F1, not just accuracy.
5. **误差分析**: 看错分样本定位原因再改进(调参/换模型/清数据)。
   Error analysis: inspect mistakes to find causes, then improve.

### 下一节 / Next
**11.5 情感分析**——文本分类的一个重要专题：判断文本的情感极性(正面/负面)。我们会对比**词典法**(查情感词典打分)和**机器学习法**，并讨论否定("not good")等真实难点。
**11.5 Sentiment Analysis** — a key specialization of text classification: judging polarity (positive/negative). We compare the **lexicon approach** (scoring with a sentiment dictionary) vs **machine learning**, and discuss real challenges like negation ("not good").
